# Project ATLAS: Google Colab GPU Campaign Runner

**Associative Topological LATent Skills (ATLAS)**: Reuses the frozen T2 world model backbone, self-learning visual skills, topological navigation, and epistemic curiosity around it.

### Storage Contract:
- **Main Google Drive folder**: `/content/drive/MyDrive/learn-a-world` (shared dataset & pre-trained checkpoints)
- **Input sequence data**: `/content/drive/MyDrive/learn-a-world/data/sequences/*.npz`
- **Frozen backbone checkpoint**: `/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt`
- **Dedicated ATLAS subfolder**: `/content/drive/MyDrive/learn-a-world/atlas/`
  - `atlas/shards/`: newly collected shards and cached `.latents.npz`
  - `atlas/runs/<run_id>/`: checkpoints (`state.pt`), `goal_bank`, `lift.npz`, `topomap.npz`, `train_log.jsonl`
  - `atlas/results/`: official benchmark screening reports (`atlas_screen_<tag>.json`)

## 1. Mount Google Drive

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/learn-a-world")
ATLAS_DIR = DRIVE_DIR / "atlas"

# Create dedicated ATLAS subfolders on Drive
for sub in ["shards", "runs", "checkpoints", "results"]:
    (ATLAS_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Google Drive mounted successfully.")
print(f"Main Drive root: {DRIVE_DIR}")
print(f"ATLAS dedicated folder: {ATLAS_DIR}")

Mounted at /content/drive
Google Drive mounted successfully.
Main Drive root: /content/drive/MyDrive/learn-a-world
ATLAS dedicated folder: /content/drive/MyDrive/learn-a-world/atlas


## 2. Sync Repository

In [2]:
%cd /content
import os

if not os.path.exists("learn-a-world"):
    !git clone https://github.com/sahiltolani30/learn-a-world.git

%cd /content/learn-a-world
!git fetch origin
!git checkout main
!git pull origin main
!git log -1 --oneline

/content
Cloning into 'learn-a-world'...
remote: Enumerating objects: 2210, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 2210 (delta 111), reused 138 (delta 61), pack-reused 2012 (from 1)
Receiving objects: 100% (2210/2210), 35.46 MiB | 14.41 MiB/s, done.
Resolving deltas: 100% (1417/1417), done.
/content/learn-a-world
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
Already up to date.
3e06e04 (HEAD -> main, origin/main, origin/HEAD) fix(mpc): only instant-craft for assembly/placement tasks; navigate with policy-seeded CEM for gathering


## 3. Environment Setup & Hardware Verification

In [3]:
%cd /content/learn-a-world
!bash atlas/colab/setup_atlas.sh
!pip install torch

/content/learn-a-world
=== [ATLAS] Setting up Project ATLAS in Colab ===
[Drive] Google Drive is mounted at /content/drive/MyDrive/learn-a-world
[Drive] Setting up dedicated ATLAS subfolder at /content/drive/MyDrive/learn-a-world/atlas...
[Drive] Dedicated ATLAS folders created successfully.
[Setup] Installing requirements...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[Setup] Champion checkpoint found on Drive: /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt
[Setup] Checking compute hardware...
CUDA Available: False | Device: CPU
=== [ATLAS] Colab Environment Setup Complete ===


## 4. Run Unit Tests & Local Smoke Verification

In [4]:
%cd /content/learn-a-world
# Run all 26 ATLAS unit tests
!python3 -m pytest atlas/tests -v

# Run end-to-end 2-episode smoke test
!python3 -m atlas.smoke

/content/learn-a-world
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/learn-a-world
configfile: pyproject.toml
plugins: platformdirs-4.13.5, langsmith-0.14.12, anyio-4.15.1, typeguard-4.6.0
collected 26 items                                                             

atlas/tests/test_atlas_arbiter.py::test_agent_act_strict_signature PASSED [  3%]
atlas/tests/test_atlas_arbiter.py::test_safety_override_fires PASSED     [  7%]
atlas/tests/test_atlas_arbiter.py::test_mode_transitions PASSED          [ 11%]
atlas/tests/test_atlas_backbone.py::test_backbone_load_and_freeze PASSED [ 15%]
atlas/tests/test_atlas_backbone.py::test_backbone_encode_determinism PASSED [ 19%]
atlas/tests/test_atlas_backbone.py::test_backbone_methods_shapes PASSED  [ 23%]
atlas/tests/test_atlas_belief.py::test_belief_threshold_update PASSED    [ 26%]
atlas/te

## 5. Check Drive Checkpoint and Sequences Availability

In [5]:
import glob
from pathlib import Path

drive_ckpt = Path("/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt")
drive_seqs = sorted(glob.glob("/content/drive/MyDrive/learn-a-world/data/sequences/*.npz"))

print(f"Champion Checkpoint exists on Drive: {drive_ckpt.exists()}")
print(f"Found {len(drive_seqs)} sequence shards on Drive for bootstrapping.")

Champion Checkpoint exists on Drive: True
Found 32 sequence shards on Drive for bootstrapping.


## 6. Phase 1: Offline Bootstrap on Drive Sequences
Reads pre-collected sequence shards from Drive, caches latents into `atlas/shards/`, trains OptionNet, ReachabilityNet, EpistemicNet, ContinueNet, and mines the initial `GoalBank` and `PredecessorLift`.

In [11]:
%cd /content/learn-a-world
!git pull origin main

%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase bootstrap \
  --iters 10 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

/content/learn-a-world
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 11 (delta 9), reused 11 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 3.07 KiB | 628.00 KiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   22c6cad..a75a612  main       -> origin/main
Updating 22c6cad..a75a612
Fast-forward
 atlas/selflearn.py                           |  14 +-
 atlas/skills.py                              |  94 +++++++----
 atlas/train.py                               |  16 +-
 godfather_jepa/memory/consolidate_offline.py | 226 +++++++++++++++------------
 4 files changed, 211 insertions(+), 139 deletions(-)
/content/learn-a-world
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the a

## 7. Phase 2: Autonomous Self-Learn Loop
Executes iterative cycles: `[Collect episodes -> Shard -> Cache latents -> Train heads -> Mine GoalBank -> Save state.pt]`.
Gracefully stops before the `--minutes` budget expires and saves state atomically directly to Drive.

In [19]:
%cd /content/learn-a-world
!git pull origin main

/content/learn-a-world
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 987 bytes | 493.00 KiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   a75a612..53a316c  main       -> origin/main
Updating a75a612..53a316c
Fast-forward
 atlas/selflearn.py | 10 ++++++++++
 atlas/train.py     | 16 +++++++++++-----
 2 files changed, 21 insertions(+), 5 deletions(-)


In [20]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase selflearn \
  --iters 3 \
  --minutes 80 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

/content/learn-a-world
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[Drive] Detected main Drive directory: /content/drive/MyDrive/learn-a-world
[ATLAS] Initializing FrozenBackbone on device 'auto'...
[JEPAv3 Checkpoint Audit] Loaded from: /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt (version: v3)
  -> Successfully Loaded (14): encoder, target_encoder, recurrent_core, predictor, event_head, health_head, state_delta_head, state_readout_head, transition_head, q_ensemble, q_target_ensemble, policy_prior, time_to_event_head, normalizer
[JEPAv3] Event-head calibrator successfully loaded from /content/learn-a-world/results/calibration/event_

In [21]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase selflearn \
  --iters 3 \
  --minutes 80 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

/content/learn-a-world
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[Drive] Detected main Drive directory: /content/drive/MyDrive/learn-a-world
[ATLAS] Initializing FrozenBackbone on device 'auto'...
[JEPAv3 Checkpoint Audit] Loaded from: /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt (version: v3)
  -> Successfully Loaded (14): encoder, target_encoder, recurrent_core, predictor, event_head, health_head, state_delta_head, state_readout_head, transition_head, q_ensemble, q_target_ensemble, policy_prior, time_to_event_head, normalizer
[JEPAv3] Event-head calibrator successfully loaded from /content/learn-a-world/results/calibration/event_

In [22]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase selflearn \
  --iters 3 \
  --minutes 80 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

/content/learn-a-world
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[Drive] Detected main Drive directory: /content/drive/MyDrive/learn-a-world
[ATLAS] Initializing FrozenBackbone on device 'auto'...
[JEPAv3 Checkpoint Audit] Loaded from: /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt (version: v3)
  -> Successfully Loaded (14): encoder, target_encoder, recurrent_core, predictor, event_head, health_head, state_delta_head, state_readout_head, transition_head, q_ensemble, q_target_ensemble, policy_prior, time_to_event_head, normalizer
[JEPAv3] Event-head calibrator successfully loaded from /content/learn-a-world/results/calibration/event_

In [23]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase selflearn \
  --iters 3 \
  --minutes 80 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

/content/learn-a-world
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[Drive] Detected main Drive directory: /content/drive/MyDrive/learn-a-world
[ATLAS] Initializing FrozenBackbone on device 'auto'...
[JEPAv3 Checkpoint Audit] Loaded from: /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt (version: v3)
  -> Successfully Loaded (14): encoder, target_encoder, recurrent_core, predictor, event_head, health_head, state_delta_head, state_readout_head, transition_head, q_ensemble, q_target_ensemble, policy_prior, time_to_event_head, normalizer
[JEPAv3] Event-head calibrator successfully loaded from /content/learn-a-world/results/calibration/event_

In [24]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase selflearn \
  --iters 3 \
  --minutes 80 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

/content/learn-a-world
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[Drive] Detected main Drive directory: /content/drive/MyDrive/learn-a-world
[ATLAS] Initializing FrozenBackbone on device 'auto'...
[JEPAv3 Checkpoint Audit] Loaded from: /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt (version: v3)
  -> Successfully Loaded (14): encoder, target_encoder, recurrent_core, predictor, event_head, health_head, state_delta_head, state_readout_head, transition_head, q_ensemble, q_target_ensemble, policy_prior, time_to_event_head, normalizer
[JEPAv3] Event-head calibrator successfully loaded from /content/learn-a-world/results/calibration/event_

In [25]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase selflearn \
  --iters 30 \
  --minutes 45 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

/content/learn-a-world
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[Drive] Detected main Drive directory: /content/drive/MyDrive/learn-a-world
[ATLAS] Initializing FrozenBackbone on device 'auto'...
[JEPAv3 Checkpoint Audit] Loaded from: /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt (version: v3)
  -> Successfully Loaded (14): encoder, target_encoder, recurrent_core, predictor, event_head, health_head, state_delta_head, state_readout_head, transition_head, q_ensemble, q_target_ensemble, policy_prior, time_to_event_head, normalizer
[JEPAv3] Event-head calibrator successfully loaded from /content/learn-a-world/results/calibration/event_

In [26]:
import os
import shutil
from pathlib import Path

# Force the operating system to flush all pending buffers to Google Drive cloud
os.sync()

print("Files currently inside your Google Drive results folder:")
!ls -lh /content/drive/MyDrive/learn-a-world/atlas/results/

print("\nFiles in your runs folder (checkpoints & memories):")
!ls -lh /content/drive/MyDrive/learn-a-world/atlas/runs/atlas_v1/

Files currently inside your Google Drive results folder:
total 14K
-rw------- 1 root root 1.8K Oct 29 11:04 atlas_screen_atlas_v1_dev_it14.json
-rw------- 1 root root 1.8K Oct 29 11:14 atlas_screen_atlas_v1_dev_it19.json
-rw------- 1 root root 1.8K Oct 29 11:18 atlas_screen_atlas_v1_dev_it24.json
-rw------- 1 root root 1.8K Oct 29 11:22 atlas_screen_atlas_v1_dev_it29.json
-rw------- 1 root root 1.8K Oct 29 11:27 atlas_screen_atlas_v1_dev_it34.json
-rw------- 1 root root 1.8K Oct 29 10:56 atlas_screen_atlas_v1_dev_it4.json
-rw------- 1 root root 1.8K Oct 29 11:00 atlas_screen_atlas_v1_dev_it9.json

Files in your runs folder (checkpoints & memories):
total 7.0M
-rw------- 1 root root  36K Oct 29 11:27 goal_bank.json
-rw------- 1 root root 665K Oct 29 11:27 goal_bank.npz
-rw------- 1 root root  611 Oct 29 11:27 lift.npz
-rw------- 1 root root 4.3M Oct 29 11:27 state.pt
-rw------- 1 root root 2.1M Oct 29 11:27 topomap.npz
-rw------- 1 root root  13K Oct 29 11:26 train_log.jsonl


In [27]:
# 1. Check if Google Drive is currently mounted
!mount | grep -i drive

# 2. Locate all saved ATLAS files across the filesystem
print("\n--- Finding state.pt checkpoints ---")
!find /content -name "state.pt" 2>/dev/null

print("\n--- Finding goal_bank.json files ---")
!find /content -name "goal_bank.json" 2>/dev/null

print("\n--- Finding screening result JSON files ---")
!find /content -name "atlas_screen_*.json" 2>/dev/null

drive on /content/drive type fuse.drive (rw,nosuid,nodev,relatime,user_id=0,group_id=0)

--- Finding state.pt checkpoints ---
/content/drive/.shortcut-targets-by-id/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa/learn-a-world/atlas/runs/atlas_v1/state.pt

--- Finding goal_bank.json files ---
/content/drive/.shortcut-targets-by-id/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa/learn-a-world/atlas/runs/atlas_v1/goal_bank.json

--- Finding screening result JSON files ---
/content/learn-a-world/results/atlas_screen_atlas_v1_dev_it14.json
/content/learn-a-world/results/atlas_screen_atlas_v1_dev_it4.json
/content/learn-a-world/results/atlas_screen_atlas_v1_dev_it9.json
/content/drive/.shortcut-targets-by-id/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa/learn-a-world/atlas/results/atlas_screen_atlas_v1_dev_it4.json
/content/drive/.shortcut-targets-by-id/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa/learn-a-world/atlas/results/atlas_screen_atlas_v1_dev_it9.json
/content/drive/.shortcut-targets-by-id/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa/learn-a-w

In [29]:
import os
from pathlib import Path
from datetime import datetime

print("=" * 65)
print("🔍 ATLAS FILE LOCATOR")
print("=" * 65)

# Target file names we want to find
TARGET_FILES = {
    "state.pt",
    "goal_bank.json",
    "goal_bank.npz",
    "lift.npz",
    "topomap.npz",
}

found = {}

# Search both Google Drive and local learn-a-world directory
for root_dir in ["/content/drive", "/content/learn-a-world"]:
    if not os.path.exists(root_dir):
        continue
    for dirpath, _, filenames in os.walk(root_dir):
        for name in filenames:
            if name in TARGET_FILES or name.startswith("atlas_screen_"):
                full_path = os.path.join(dirpath, name)
                real_path = os.path.realpath(full_path)
                if real_path not in found:
                    stat = os.stat(real_path)
                    found[real_path] = {
                        "name": name,
                        "dir": os.path.dirname(real_path),
                        "size": f"{stat.st_size / (1024 * 1024):.2f} MB" if stat.st_size > 1024*1024 else f"{stat.st_size / 1024:.1f} KB",
                        "time": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
                    }

# Group files by directory
dirs = {}
for real_path, meta in found.items():
    dirs.setdefault(meta["dir"], []).append(meta)

if dirs:
    print(f"\n✅ Found {len(found)} ATLAS files across {len(dirs)} directories:\n")
    for d, files in sorted(dirs.items()):
        print(f"📁 Directory: {d}")

        # Check if this folder has a shortcut ID and show Google Drive web link
        if ".shortcut-targets-by-id" in d:
            parts = d.split(os.sep)
            if ".shortcut-targets-by-id" in parts:
                idx = parts.index(".shortcut-targets-by-id") + 1
                if idx < len(parts):
                    folder_id = parts[idx]
                    print(f"   🌐 Drive Link: https://drive.google.com/drive/folders/{folder_id}")

        for f in sorted(files, key=lambda x: x["name"]):
            print(f"   └── {f['name']:<35} ({f['size']:>9})  [{f['time']}]")
        print()
else:
    print("❌ No ATLAS files found.")

🔍 ATLAS FILE LOCATOR

✅ Found 15 ATLAS files across 3 directories:

📁 Directory: /content/drive/.shortcut-targets-by-id/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa/learn-a-world/atlas/results
   🌐 Drive Link: https://drive.google.com/drive/folders/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa
   └── atlas_screen_atlas_v1_dev_it14.json (   1.8 KB)  [2026-10-29 11:04:19]
   └── atlas_screen_atlas_v1_dev_it19.json (   1.8 KB)  [2026-10-29 11:14:38]
   └── atlas_screen_atlas_v1_dev_it24.json (   1.8 KB)  [2026-10-29 11:18:46]
   └── atlas_screen_atlas_v1_dev_it29.json (   1.8 KB)  [2026-10-29 11:22:53]
   └── atlas_screen_atlas_v1_dev_it34.json (   1.8 KB)  [2026-10-29 11:27:39]
   └── atlas_screen_atlas_v1_dev_it4.json  (   1.8 KB)  [2026-10-29 10:56:00]
   └── atlas_screen_atlas_v1_dev_it9.json  (   1.8 KB)  [2026-10-29 11:00:09]

📁 Directory: /content/drive/.shortcut-targets-by-id/18tW0Bzv4yPHA5kCgtSjS-ytI4IDUYJaa/learn-a-world/atlas/runs/atlas_v1
   🌐 Drive Link: https://drive.google.com/drive/folders/18tW0

## 8. Phase 3: Locked 50-Episode Benchmark Screening
Evaluates the trained ATLAS agent under strict locked protocol: seeds 2050–2099, 50 episodes, max 1000 steps, zero exploration (epsilon=0, beta=0). Results are written to `results/` and `/content/drive/MyDrive/learn-a-world/atlas/results/`.

In [ ]:
%cd /content/learn-a-world
!python3 -m atlas.colab.run_atlas \
  --phase screen \
  --tag atlas_v1 \
  --run-id atlas_v1 \
  --drive-dir /content/drive/MyDrive/learn-a-world

## 9. Inspect Screening Results & Achievements

In [ ]:
import json
from pathlib import Path

res_file = Path("/content/drive/MyDrive/learn-a-world/atlas/results/atlas_screen_atlas_v1.json")
if not res_file.exists():
    res_file = Path("results/atlas_screen_atlas_v1.json")

if res_file.exists():
    with open(res_file) as f:
        data = json.load(f)
    print(f"Official Crafter Score: {data['crafter_score']:.2f}%")
    print(f"Mean Survival: {data['mean_survival']:.1f} steps | Mean Reward: {data['mean_reward']:.2f}")
    print(f"Telemetry: {data['per_mode_telemetry']}")
    print("\nKey Milestone Unlocks (% of episodes):")
    for k in ["collect_wood", "place_table", "make_wood_pickaxe", "collect_stone", "make_stone_pickaxe", "collect_coal", "place_furnace", "collect_iron"]:
        print(f"  {k:20s}: {data['unlock_percentages'].get(k, 0.0):.1f}%")
else:
    print("No screening report found yet.")

In [18]:
import json
import numpy as np
from pathlib import Path
from eval.metrics import CRAFTER_ACHIEVEMENTS

run_dir = Path("/content/drive/MyDrive/learn-a-world/atlas/runs/atlas_v1")

# 1. GoalBank Audit
with open(run_dir / "goal_bank.json") as f:
    goals = json.load(f)

print(f"==================================================")
print(f"=== [ATLAS Memory Audit: {len(goals)} Goals] ===")
print(f"==================================================")

counts_by_kind = {}
event_ach_counts = {}

for g in goals:
    k = g["kind"]
    counts_by_kind[k] = counts_by_kind.get(k, 0) + 1
    for dim in g["event_dims"]:
        ach_name = CRAFTER_ACHIEVEMENTS[dim]
        event_ach_counts[ach_name] = event_ach_counts.get(ach_name, 0) + 1

print("\n1. Goals by Category:")
for k, cnt in counts_by_kind.items():
    print(f"  • {k.capitalize()} goals: {cnt}")

print("\n2. Achievement Landmarks Memorized in Memory:")
for ach, cnt in sorted(event_ach_counts.items(), key=lambda x: -x[1]):
    print(f"  • {ach}: {cnt} visual landmark states")

# 2. Causal Lift Audit (Who precedes who?)
lift_data = np.load(run_dir / "lift.npz")
lift_mat = lift_data["lift"]

print("\n3. Top Discovered Causal Prerequisite Rules (Lift > 2.0x):")
found_rules = []
for e in range(len(CRAFTER_ACHIEVEMENTS)):
    for d in range(len(CRAFTER_ACHIEVEMENTS)):
        val = lift_mat[d, e]
        if val > 2.0 and d != e:
            found_rules.append((val, CRAFTER_ACHIEVEMENTS[d], CRAFTER_ACHIEVEMENTS[e]))

found_rules.sort(key=lambda x: -x[0])
for val, prereq, target in found_rules[:15]:
    print(f"  • [{prereq}] is a proven prerequisite for [{target}] (Lift: {val:.1f}x)")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


=== [ATLAS Memory Audit: 256 Goals] ===

1. Goals by Category:
  • Event goals: 197
  • Frontier goals: 59

2. Achievement Landmarks Memorized in Memory:
  • collect_sapling: 116 visual landmark states
  • collect_drink: 25 visual landmark states
  • collect_wood: 22 visual landmark states
  • place_plant: 17 visual landmark states
  • wake_up: 13 visual landmark states
  • eat_cow: 6 visual landmark states
  • defeat_zombie: 5 visual landmark states
  • place_table: 4 visual landmark states

3. Top Discovered Causal Prerequisite Rules (Lift > 2.0x):
  • [eat_cow] is a proven prerequisite for [collect_sapling] (Lift: 36.4x)
  • [collect_sapling] is a proven prerequisite for [eat_cow] (Lift: 12.1x)
  • [collect_sapling] is a proven prerequisite for [place_plant] (Lift: 12.1x)
  • [collect_sapling] is a proven prerequisite for [defeat_zombie] (Lift: 9.1x)
  • [defeat_zombie] is a proven prerequisite for [collect_sapling] (Lift: 3.0x)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
import os
import shutil
from pathlib import Path
from google.colab import drive

dest = Path("/content/drive/MyDrive/learn-a-world/atlas")
(dest / "results").mkdir(parents=True, exist_ok=True)
(dest / "runs" / "atlas_v1").mkdir(parents=True, exist_ok=True)

# Also ensure main results and checkpoints folders exist (where other agents save)
main_results = Path("/content/drive/MyDrive/learn-a-world/results")
main_ckpts = Path("/content/drive/MyDrive/learn-a-world/checkpoints")
main_results.mkdir(parents=True, exist_ok=True)
main_ckpts.mkdir(parents=True, exist_ok=True)

target_names = {
    "state.pt", "goal_bank.json", "goal_bank.npz",
    "lift.npz", "topomap.npz", "train_log.jsonl"
}

def safe_copy(src: Path, dst: Path):
    if src.resolve() != dst.resolve():
        try:
            shutil.copy2(src, dst)
            print(f"   📋 Copied: {src.name} -> {dst}")
        except Exception as e:
            pass

print("1. Syncing ATLAS reports and checkpoints...")
for search_root in ["/content/drive", "/content/learn-a-world"]:
    if not os.path.exists(search_root):
        continue
    for root, _, files in os.walk(search_root):
        for f in files:
            full_src = Path(root) / f

            # Screening reports
            if f.startswith("atlas_screen_") and f.endswith(".json"):
                safe_copy(full_src, dest / "results" / f)
                safe_copy(full_src, dest / f)
                safe_copy(full_src, main_results / f)  # also in main results

            # Model checkpoints and memories
            elif f in target_names:
                safe_copy(full_src, dest / "runs" / "atlas_v1" / f)
                safe_copy(full_src, dest / f)
                if f == "state.pt":
                    safe_copy(full_src, main_ckpts / "atlas_v1_state.pt")

print("\n2. ☁️ Flushing Colab cache to Google Drive cloud servers (takes ~15-30 seconds)...")
drive.flush_and_unmount()

print("\n3. 🔄 Re-mounting Drive cleanly...")
drive.mount("/content/drive")

print("\n🎉 SYNC COMPLETE! Refresh your Google Drive browser tab—the files are now live in the cloud!")

1. Syncing ATLAS reports and checkpoints...
   📋 Copied: atlas_screen_atlas_v1_dev_it4.json -> /content/drive/MyDrive/learn-a-world/atlas/results/atlas_screen_atlas_v1_dev_it4.json
   📋 Copied: atlas_screen_atlas_v1_dev_it4.json -> /content/drive/MyDrive/learn-a-world/atlas/atlas_screen_atlas_v1_dev_it4.json
   📋 Copied: atlas_screen_atlas_v1_dev_it9.json -> /content/drive/MyDrive/learn-a-world/atlas/results/atlas_screen_atlas_v1_dev_it9.json
   📋 Copied: atlas_screen_atlas_v1_dev_it9.json -> /content/drive/MyDrive/learn-a-world/atlas/atlas_screen_atlas_v1_dev_it9.json
   📋 Copied: atlas_screen_atlas_v1_dev_it14.json -> /content/drive/MyDrive/learn-a-world/atlas/results/atlas_screen_atlas_v1_dev_it14.json
   📋 Copied: atlas_screen_atlas_v1_dev_it14.json -> /content/drive/MyDrive/learn-a-world/atlas/atlas_screen_atlas_v1_dev_it14.json
   📋 Copied: atlas_screen_atlas_v1_dev_it19.json -> /content/drive/MyDrive/learn-a-world/atlas/results/atlas_screen_atlas_v1_dev_it19.json
   📋 Copied: at